In [3]:
import numpy as np

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

In [5]:

# =========================
# STEP 1: LOAD DATA
# =========================
df = pd.read_csv("market_sales_prediction.csv")

print("Dataset Shape:", df.shape)
print(df.head())

# =========================
# STEP 2: TARGET DETECT
# =========================
# Last column as target (auto)
target_col = df.columns[-1]
print("Target Column:", target_col)

# =========================
# STEP 3: MISSING VALUES
for col in df.columns:
    
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    
    else:
        # convert safely to numeric
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].mean())
# =========================
# STEP 4: FEATURE ENGINEERING (AUTO)
# =========================
# Example: create interaction features (optional)
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    if df[col].min() >= 0:
        df[col+"_log"] = np.log1p(df[col])

# =========================
# STEP 5: ENCODING
# =========================
le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

# =========================
# STEP 6: SPLIT
# =========================
X = df.drop(target_col, axis=1)
y = df[target_col]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# STEP 7: SCALING
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)

# =========================
# STEP 8: RANDOM FOREST
# =========================
rf = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_valid)

# =========================
# STEP 9: XGBOOST
# =========================
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_valid)

# =========================
# STEP 10: EVALUATION
# =========================
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name} RMSE:", rmse)
    print(f"{name} R2:", r2)
    print("-"*40)

evaluate("Random Forest", y_valid, rf_pred)
evaluate("XGBoost", y_valid, xgb_pred)

# =========================
# STEP 11: BEST MODEL
# =========================
if r2_score(y_valid, xgb_pred) > r2_score(y_valid, rf_pred):
    final_model = xgb
    print("Using XGBoost ✅")
else:
    final_model = rf
    print("Using Random Forest ✅")

# =========================
# STEP 12: FINAL TRAIN ON FULL DATA
# =========================
X_scaled = scaler.fit_transform(X)
final_model.fit(X_scaled, y)

print("🔥 Model Ready Successfully!")

Dataset Shape: (16160, 19)
   Order_ID        Date Region       City Product_Category  Product_Name  \
0    101800  03/06/2021   EAST      Patna      Electronics   Lapttop Pro   
1    106625  2021-05-22   West      Surat        Furniture  Wooden Chair   
2    113555  2022-03-16   West  Ahmedabad        Furniture      Sofa Set   
3    102309  2021-03-29   West       Pune      Electronics     iPhone 14   
4    107837  29/11/2020  South  Hyderabad          Grocery   Wheat Flour   

     Price  Discount (%)  Quantity  Customer_Age Customer_Gender  \
0   217.92          29.1       5.0            20            MALE   
1  1480.85          31.5       1.0            31          female   
2   365.34          21.2      10.0            48            Male   
3  1800.11           NaN       7.0            41          Female   
4    40.90          25.1       1.0            37          female   

  Sales_Channel Payment_Method   Season  Marketing_Spend  Competitor_Price  \
0        Online    Credit Car

In [7]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [8]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "Decision Tree": DecisionTreeRegressor(max_depth=10),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200),
    "AdaBoost": AdaBoostRegressor(n_estimators=200),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05)
}

In [10]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    
    rmse = np.sqrt(mean_squared_error(y_valid, pred))
    r2 = r2_score(y_valid, pred)
    
    results.append((name, rmse, r2))
    
    print(f"{name}")
    print("RMSE:", rmse)
    print("R2 Score:", r2)
    print("-"*40)

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [13]:
# =========================================
# 🔥 COMPLETE ML PIPELINE (FINAL FIXED)
# =========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

# =========================================
# STEP 1: LOAD DATA
# =========================================
df = pd.read_csv("market_sales_prediction.csv")

print("Dataset Shape:", df.shape)

# =========================================
# STEP 2: TARGET
# =========================================
target_col = df.columns[-1]

# =========================================
# STEP 3: CLEANING (FIXED VERSION 🔥)
# =========================================
for col in df.columns:
    
    if df[col].dtype == "object":
        try:
            # try converting to numeric
            df[col] = pd.to_numeric(df[col])
        except:
            pass

    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

# =========================================
# STEP 4: FEATURE ENGINEERING
# =========================================
for col in df.select_dtypes(include=np.number).columns:
    if col != target_col:
        if (df[col] >= 0).all():
            df[col + "_log"] = np.log1p(df[col])

# =========================================
# STEP 5: ENCODING
# =========================================
le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

# =========================================
# STEP 6: FINAL NaN FIX
# =========================================
df = df.fillna(0)

# =========================================
# STEP 7: SPLIT
# =========================================
X = df.drop(target_col, axis=1)
y = df[target_col]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================
# STEP 8: SCALING
# =========================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)

# EXTRA SAFETY
X_train = np.nan_to_num(X_train)
X_valid = np.nan_to_num(X_valid)

# =========================================
# STEP 9: MODELS
# =========================================
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Decision Tree": DecisionTreeRegressor(max_depth=10),
    "Random Forest": RandomForestRegressor(n_estimators=200),
    "Gradient Boosting": GradientBoostingRegressor(),
    "AdaBoost": AdaBoostRegressor(),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor(),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05)
}

# =========================================
# STEP 10: TRAIN + EVALUATE
# =========================================
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    
    rmse = np.sqrt(mean_squared_error(y_valid, pred))
    r2 = r2_score(y_valid, pred)
    
    results.append((name, rmse, r2))
    
    print(f"\n{name}")
    print("RMSE:", rmse)
    print("R2:", r2)

# =========================================
# STEP 11: BEST MODEL
# =========================================
results_df = pd.DataFrame(results, columns=["Model", "RMSE", "R2"])
results_df = results_df.sort_values(by="R2", ascending=False)

print("\n🔥 BEST MODEL:")
print(results_df.iloc[0])

best_model = models[results_df.iloc[0]["Model"]]

# =========================================
# STEP 12: FINAL TRAIN
# =========================================
X_scaled = scaler.fit_transform(X)
X_scaled = np.nan_to_num(X_scaled)

best_model.fit(X_scaled, y)

print("\n✅ MODEL READY SUCCESSFULLY!")

Dataset Shape: (16160, 19)

Linear
RMSE: 1944.2902537814484
R2: 0.22450534542663958

Ridge
RMSE: 1944.349764158881
R2: 0.22445787238523052

Lasso
RMSE: 1944.0735893936162
R2: 0.2246781722074146

Decision Tree
RMSE: 2273.6485833001148
R2: -0.060481975980937186

Random Forest
RMSE: 1995.3061255001521
R2: 0.1832753153303157

Gradient Boosting
RMSE: 1951.252443449519
R2: 0.21894155927143677

AdaBoost
RMSE: 2165.699046089701
R2: 0.037827764458986946

SVR
RMSE: 2403.376464941752
R2: -0.18495052663018008

KNN
RMSE: 2063.018498770345
R2: 0.12690227873328874

XGBoost
RMSE: 1996.1386648063603
R2: 0.18259361817219888

🔥 BEST MODEL:
Model          Lasso
RMSE     1944.073589
R2          0.224678
Name: 2, dtype: object

✅ MODEL READY SUCCESSFULLY!


In [14]:
df = pd.read_csv('train.csv')
df.columns

Index(['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility',
       'Item_Type', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Outlet_Sales'],
      dtype='str')